# Logistic Regression Scorecard (WoE / IV)

This notebook walks through the full credit scorecard construction pipeline applied to fraud detection. The methodology is the industry standard used at banks and credit bureaus for consumer risk models (Siddiqi 2006) and is explicitly referenced in APRA CPG 220.

## Why scorecards?

| Property | Isolation Forest | XGBoost | **Scorecard** |
|---|---|---|---|
| Supervised | ✗ | ✓ | **✓** |
| Interpretable | ✗ | Partial (SHAP) | **Fully additive** |
| Regulator-friendly | ✗ | Partial | **✓ (APRA/ASIC/Basel)** |
| Requires fraud labels | ✗ | ✓ | **✓** |
| Handles imbalance | ✓ | ✓ (scale_pos_weight) | **✓ (class_weight)** |
| Bin-level audit trail | ✗ | ✗ | **✓** |

The additive points structure means any analyst can reproduce a score from the scorecard table in a spreadsheet — no black box.

## Pipeline overview
1. Feature engineering (is_night)
2. **Information Value (IV)** screening — remove uninformative features
3. **Weight of Evidence (WoE)** binning — transform raw values
4. WoE bin plots — validate monotonicity
5. Logistic Regression on WoE features
6. **PDO scaling** — map log-odds to integer points
7. **Scorecard table** — feature × bin → points
8. Score distribution & separation
9. Gini, KS statistic, AUC-PR, AUC-ROC
10. Score banding — Decline / Review / Pass

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
)
from sklearn.model_selection import train_test_split

from src.data.loader import load_transactions
from src.models.scorecard import (
    WoEBinner, LRScoreCard, train_scorecard,
    gini_from_auc, ks_statistic,
    CATEGORICAL_FEATURES,
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

TIER_COLOURS = {
    'LOW':      '#22c55e',
    'MEDIUM':   '#f59e0b',
    'HIGH':     '#ef4444',
    'CRITICAL': '#7c3aed',
}

print('Setup complete.')

## 2. Load & Prepare Data

In [ ]:
df = load_transactions()

# Derived feature used by the scorecard
df['is_night'] = ((df['hour'] >= 23) | (df['hour'] <= 5)).astype(int)

print(f'Total transactions: {len(df):,}')
print(f'Fraud rate        : {df["is_fraud"].mean():.2%}')
print(f'\nFeature columns   : {[c for c in df.columns if c != "is_fraud"]}')

In [ ]:
from src.data.features import SYNTHETIC_NUMERIC_FEATURES

FEATURE_COLS = [f for f in SYNTHETIC_NUMERIC_FEATURES + ['is_night'] if f in df.columns]
X = df[FEATURE_COLS].copy()
y = df['is_fraud'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train fraud rate: {y_train.mean():.2%} | Test: {y_test.mean():.2%}')

---
## 3. Information Value (IV) Screening

IV measures each feature's predictive power on its own:

$$IV_i = \sum_{j=1}^{B} \left(\frac{n_{bad,j}}{N_{bad}} - \frac{n_{good,j}}{N_{good}}\right) \cdot WoE_j$$

**Siddiqi (2006) IV interpretation:**

| IV | Predictive Power |
|---|---|
| < 0.02 | Useless |
| 0.02 – 0.10 | Weak |
| 0.10 – 0.30 | Medium |
| 0.30 – 0.50 | Strong |
| > 0.50 | Suspicious (check for leakage) |

In [ ]:
# Fit binner on ALL features first to rank by IV
binner_full = WoEBinner(n_bins=8, min_bads=3)
binner_full.fit(X_train, y_train)

iv_all = binner_full.get_iv_summary()
print('Information Value — all features:')
print(iv_all.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

iv_sorted = iv_all.sort_values('iv')
colours = [
    '#ef4444' if iv > 0.5 else
    '#3b82f6' if iv > 0.3 else
    '#6366f1' if iv > 0.1 else
    '#94a3b8' if iv > 0.02 else
    '#e2e8f0'
    for iv in iv_sorted['iv']
]

bars = ax.barh(iv_sorted['feature'], iv_sorted['iv'], color=colours, edgecolor='white')
ax.axvline(0.02, color='#f59e0b', linestyle='--', linewidth=1.5, label='Weak (0.02)')
ax.axvline(0.10, color='#6366f1', linestyle='--', linewidth=1.5, label='Medium (0.10)')
ax.axvline(0.30, color='#3b82f6', linestyle='--', linewidth=1.5, label='Strong (0.30)')

for bar, iv in zip(bars, iv_sorted['iv']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{iv:.4f}', va='center', fontsize=9)

ax.set_xlabel('Information Value (IV)')
ax.set_title('Feature Predictive Power — Information Value', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
MIN_IV = 0.02
selected_features = iv_all[iv_all['iv'] >= MIN_IV]['feature'].tolist()
dropped_features  = iv_all[iv_all['iv'] <  MIN_IV]['feature'].tolist()

print(f'Selected ({len(selected_features)}): {selected_features}')
print(f'Dropped  ({len(dropped_features)}): {dropped_features}')

---
## 4. Weight of Evidence (WoE) Binning

For each bin $j$ of feature $i$:

$$WoE_j = \ln\left(\frac{\% Fraud_j}{\% Legitimate_j}\right)$$

Positive WoE → that bin is over-represented in fraud. Negative → under-represented.

Monotonic WoE across bins is the "coarse classing" quality check.

In [ ]:
# Fit binner on selected features
binner = WoEBinner(n_bins=8, min_bads=3)
binner.fit(X_train[selected_features], y_train)

woe_table = binner.get_woe_table()
print('WoE table (first 20 rows):')
print(woe_table.head(20).to_string(index=False))

In [ ]:
def plot_woe_bins(binner, features, ncols=3):
    """Grid of WoE bar charts — one per feature."""
    n     = len(features)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.array(axes).flatten()

    for ax, feat in zip(axes, features):
        info   = binner.feature_info_[feat]
        tbl    = info['table']
        labels = tbl['bin'].astype(str)
        woes   = tbl['woe'].values
        ev_r   = tbl['event_rate'].values

        cols = ['#ef4444' if w > 0 else '#3b82f6' for w in woes]
        ax.bar(range(len(labels)), woes, color=cols, edgecolor='white', width=0.7)
        ax.axhline(0, color='black', linewidth=0.8)

        # Overlay event rate as text
        for i, (w, er) in enumerate(zip(woes, ev_r)):
            y_pos = w + 0.02 if w >= 0 else w - 0.08
            ax.text(i, y_pos, f'{er:.1%}', ha='center', fontsize=7)

        ax.set_title(feat, fontweight='bold', fontsize=10)
        ax.set_ylabel('WoE')
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
        iv_val = info['iv']
        ax.set_xlabel(f'IV = {iv_val:.4f}', fontsize=9)

    # Hide unused axes
    for ax in axes[n:]:
        ax.set_visible(False)

    plt.suptitle('WoE by Bin (red = fraud-elevated, blue = fraud-suppressed)', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_woe_bins(binner, selected_features)

---
## 5. WoE Transformation

In [ ]:
X_train_woe = binner.transform(X_train[selected_features])
X_test_woe  = binner.transform(X_test[selected_features])

print('WoE-transformed training sample (first 5 rows):')
X_train_woe.head()

In [ ]:
# Verify WoE-transformed data separates the classes
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Correlation heatmap of WoE features
sns.heatmap(X_train_woe.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[0], linewidths=0.5, annot_kws={'size': 7})
axes[0].set_title('WoE Feature Correlations (train)', fontweight='bold')
axes[0].tick_params(labelsize=8)

# WoE distribution: fraud vs legit for the top IV feature
top_feat = iv_all.iloc[0]['feature']
for cls, col, lbl in [(0, '#3b82f6', 'Legitimate'), (1, '#ef4444', 'Fraud')]:
    mask = (y_train == cls)
    sns.kdeplot(X_train_woe.loc[mask.values, top_feat],
                ax=axes[1], color=col, fill=True, alpha=0.4, label=lbl, linewidth=2)
axes[1].set_xlabel(f'WoE({top_feat})')
axes[1].set_title(f'WoE Distribution: {top_feat}', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6. Fit Logistic Regression Scorecard

In [ ]:
card = LRScoreCard(base_score=600, base_odds=50, pdo=20, C=1.0)
card.fit(X_train_woe, y_train, binner)

print(f'Intercept (α) : {card.lr_.intercept_[0]:.4f}')
print(f'Scaling B     : {card.B_:.4f}  (PDO / log(2))')
print(f'Scaling A     : {card.A_:.4f}  (base_score + B*log(base_odds))')
print(f'Offset        : {card.offset_} pts  (intercept contribution)')
print()
print('Feature coefficients:')
coef_df = pd.DataFrame({
    'feature': list(card.feature_coefs_.keys()),
    'coef':    list(card.feature_coefs_.values()),
}).sort_values('coef', ascending=False)
print(coef_df.to_string(index=False))

---
## 7. The Scorecard Table

This is the key deliverable of the scorecard approach. Every combination of feature × bin maps to a fixed number of integer points. Higher points = more fraud risk. An analyst can score any transaction by hand using this table.

In [ ]:
sc_table = card.scorecard_table_.copy()
print('Full Scorecard Table:')
print('=' * 90)
print(sc_table[['feature','bin','n_total','n_bad','event_rate','woe','points']]
      .to_string(index=False))

In [ ]:
# Visualise points contribution per feature
detail = sc_table[sc_table['feature'] != '__OFFSET__'].copy()

# Points range per feature
feat_range = (
    detail.groupby('feature')['points']
    .agg(min_pts='min', max_pts='max')
    .assign(range=lambda d: d['max_pts'] - d['min_pts'])
    .sort_values('range', ascending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Points range per feature (how much each feature can swing the total)
colours = ['#ef4444' if r > 0 else '#3b82f6' for r in feat_range['range']]
axes[0].barh(feat_range.index, feat_range['range'],
             color='#6366f1', edgecolor='white')
axes[0].set_xlabel('Score Points Range (max - min across bins)')
axes[0].set_title('Feature Contribution Range', fontweight='bold')
axes[0].set_xlabel('Total Point Swing')

# Offset + max contribution breakdown (stacked bar)
top_5 = feat_range.sort_values('range', ascending=False).head(5)
x_pos = range(len(top_5))
min_pts = top_5['min_pts'].values
ranges  = top_5['range'].values
bars_base = axes[1].bar(x_pos, min_pts, color='#3b82f6', edgecolor='white',
                        label='Min points (lowest-risk bin)')
bars_rng  = axes[1].bar(x_pos, ranges, bottom=min_pts, color='#ef4444',
                        edgecolor='white', label='Points uplift (to max bin)')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(top_5.index, rotation=20, ha='right')
axes[1].set_ylabel('Score Points')
axes[1].set_title('Top 5 Features: Points Range', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].axhline(0, color='black', linewidth=0.8)

plt.suptitle('Scorecard Points Decomposition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Score Distribution & Separation

A good scorecard separates the two classes clearly. We want the fraud score distribution to be shifted right (higher scores) compared to legitimate transactions.

In [ ]:
scores_test = card.predict_score(X_test_woe)
probs_test  = card.predict_proba(X_test_woe)

fraud_scores = scores_test[y_test.values == 1]
legit_scores = scores_test[y_test.values == 0]

print(f'Score range (test)   : {scores_test.min()} – {scores_test.max()}')
print(f'Fraud   median score : {np.median(fraud_scores):.0f}')
print(f'Legit   median score : {np.median(legit_scores):.0f}')
print(f'Separation gap       : {np.median(fraud_scores) - np.median(legit_scores):.0f} pts')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Score distributions
bins_range = np.linspace(scores_test.min(), scores_test.max(), 40)
axes[0].hist(legit_scores, bins=bins_range, color='#3b82f6', alpha=0.6,
             density=True, label='Legitimate', edgecolor='white')
axes[0].hist(fraud_scores, bins=bins_range, color='#ef4444', alpha=0.6,
             density=True, label='Fraud', edgecolor='white')
axes[0].axvline(np.median(legit_scores), color='#3b82f6', linestyle='--', linewidth=2)
axes[0].axvline(np.median(fraud_scores), color='#ef4444', linestyle='--', linewidth=2)
axes[0].set_xlabel('Scorecard Score')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distribution', fontweight='bold')
axes[0].legend()

# Fraud rate by score decile
decile_df = pd.DataFrame({'score': scores_test, 'is_fraud': y_test.values})
decile_df['decile'] = pd.qcut(decile_df['score'], q=10, labels=False, duplicates='drop') + 1
decile_stats = (
    decile_df.groupby('decile')
    .agg(n=('is_fraud','count'), n_fraud=('is_fraud','sum'), avg_score=('score','mean'))
    .assign(fraud_rate=lambda d: d['n_fraud'] / d['n'])
).reset_index()

bars = axes[1].bar(decile_stats['decile'], decile_stats['fraud_rate'] * 100,
                   color='#6366f1', edgecolor='white')
axes[1].set_xlabel('Score Decile (1=lowest, 10=highest risk)')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Score Decile', fontweight='bold')
axes[1].axhline(y_test.mean() * 100, color='grey', linestyle='--',
                linewidth=1.2, label='Overall avg')
axes[1].legend(fontsize=9)

# Cumulative separation (Lorenz curve style)
sort_idx   = np.argsort(-scores_test)  # descending score
cum_fraud  = np.cumsum(y_test.values[sort_idx]) / y_test.sum()
cum_pop    = np.arange(1, len(scores_test)+1) / len(scores_test)
axes[2].plot(cum_pop * 100, cum_fraud * 100, color='#6366f1', linewidth=2.5,
             label='Scorecard')
axes[2].plot([0, 100], [0, 100], 'k--', linewidth=1, label='Random')
axes[2].fill_between(cum_pop * 100, cum_fraud * 100, cum_pop * 100,
                     alpha=0.15, color='#6366f1')
axes[2].set_xlabel('% Population (scored high → low risk)')
axes[2].set_ylabel('% Fraud Captured')
axes[2].set_title('Lorenz / Gains Curve', fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Score Separation Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Model Performance Metrics

Standard credit risk model evaluation (APRA CPG 220 / Basel II/III validation framework):

- **Gini** = 2 × AUC − 1. Industry standard for scorecard discrimination. Target: > 0.40 (fair) / > 0.60 (good).
- **KS statistic**: maximum separation between cumulative fraud vs. legit distributions. Target: > 0.30.
- **AUC-PR**: preferred for imbalanced datasets — measures precision–recall tradeoff at all thresholds.
- **AUC-ROC**: standard but optimistic on imbalanced data; report alongside AUC-PR.

In [ ]:
auc_roc = roc_auc_score(y_test, probs_test)
auc_pr  = average_precision_score(y_test, probs_test)
gini    = gini_from_auc(auc_roc)
ks      = ks_statistic(y_test.values, probs_test)

print('=' * 50)
print('  Scorecard Performance Metrics (Test Set)')
print('=' * 50)
print(f'  AUC-PR  (primary): {auc_pr:.4f}')
print(f'  AUC-ROC           : {auc_roc:.4f}')
print(f'  Gini coefficient  : {gini:.4f}  [{"Good" if gini > 0.6 else "Fair" if gini > 0.4 else "Marginal"}]')
print(f'  KS statistic      : {ks:.4f}  [{"Good" if ks > 0.4 else "Fair" if ks > 0.3 else "Marginal"}]')
print('=' * 50)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, probs_test)
axes[0].plot(fpr, tpr, color='#6366f1', linewidth=2.5,
             label=f'LR Scorecard (AUC={auc_roc:.3f}, Gini={gini:.3f})')
axes[0].plot([0,1],[0,1],'k--', linewidth=1, label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#6366f1')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend(fontsize=9)

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, probs_test)
axes[1].plot(rec, prec, color='#ef4444', linewidth=2.5,
             label=f'LR Scorecard (AUC-PR={auc_pr:.3f})')
baseline = y_test.mean()
axes[1].axhline(baseline, color='grey', linestyle='--', linewidth=1.2,
                label=f'Baseline (prevalence={baseline:.2%})')
axes[1].fill_between(rec, prec, baseline, alpha=0.1, color='#ef4444')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend(fontsize=9)

# KS plot: fraud vs legit cumulative distributions
fraud_idx = np.where(y_test.values == 1)[0]
legit_idx = np.where(y_test.values == 0)[0]
thresholds = np.linspace(0, 1, 200)
cum_fraud_th = [np.mean(probs_test[fraud_idx] <= t) for t in thresholds]
cum_legit_th = [np.mean(probs_test[legit_idx] <= t) for t in thresholds]
ks_gap = np.abs(np.array(cum_fraud_th) - np.array(cum_legit_th))
ks_t   = thresholds[np.argmax(ks_gap)]

axes[2].plot(thresholds, cum_legit_th, color='#3b82f6', linewidth=2.5,
             label=f'Legitimate CDF')
axes[2].plot(thresholds, cum_fraud_th, color='#ef4444', linewidth=2.5,
             label=f'Fraud CDF')
axes[2].axvline(ks_t, color='grey', linestyle='--', linewidth=1.2)
axes[2].annotate(f'KS={ks:.3f}', xy=(ks_t, 0.5), xytext=(ks_t + 0.05, 0.45),
                 fontsize=10, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='grey'))
axes[2].set_xlabel('P(Fraud) Threshold')
axes[2].set_ylabel('Cumulative Proportion')
axes[2].set_title('KS Separation Plot', fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Scorecard Discrimination Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Score Banding — Operational Deployment

Scorecards are operationalised through **score bands** that define cutoff actions: Decline, Review Queue, Approve. The cutoffs are business decisions balancing fraud loss vs. friction.

Here we derive bands from probability thresholds consistent with the project's existing risk tier logic.

In [ ]:
tiers_test = card.score_to_tier(probs_test)
tier_df    = pd.DataFrame({
    'score': scores_test,
    'prob':  probs_test,
    'tier':  tiers_test,
    'actual_fraud': y_test.values,
})

band_stats = (
    tier_df.groupby('tier')
    .agg(
        count=('actual_fraud','count'),
        fraud_caught=('actual_fraud','sum'),
        min_score=('score','min'),
        max_score=('score','max'),
        avg_score=('score','mean'),
    )
    .assign(
        pct_population=lambda d: d['count'] / d['count'].sum() * 100,
        fraud_rate=lambda d: d['fraud_caught'] / d['count'] * 100,
        pct_fraud_captured=lambda d: d['fraud_caught'] / d['fraud_caught'].sum() * 100,
    )
)
tier_order = ['LOW','MEDIUM','HIGH','CRITICAL']
band_stats = band_stats.reindex([t for t in tier_order if t in band_stats.index])

print('Score Band Summary:')
print('=' * 90)
print(band_stats[['count','pct_population','fraud_caught','fraud_rate',
                   'pct_fraud_captured','avg_score']].round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

tiers_present = [t for t in tier_order if t in band_stats.index]
colours       = [TIER_COLOURS[t] for t in tiers_present]

# Population by band
axes[0].bar(tiers_present, band_stats.loc[tiers_present, 'pct_population'],
            color=colours, edgecolor='white')
axes[0].set_ylabel('% of Population')
axes[0].set_title('Population per Band', fontweight='bold')
for bar, v in zip(axes[0].patches, band_stats.loc[tiers_present,'pct_population']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{v:.1f}%', ha='center', fontweight='bold', fontsize=9)

# Fraud rate per band
axes[1].bar(tiers_present, band_stats.loc[tiers_present, 'fraud_rate'],
            color=colours, edgecolor='white')
axes[1].axhline(y_test.mean() * 100, color='grey', linestyle='--',
                linewidth=1.2, label='Overall avg')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate per Band', fontweight='bold')
axes[1].legend(fontsize=9)
for bar, v in zip(axes[1].patches, band_stats.loc[tiers_present,'fraud_rate']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{v:.1f}%', ha='center', fontweight='bold', fontsize=9)

# % Fraud captured per band
axes[2].bar(tiers_present, band_stats.loc[tiers_present,'pct_fraud_captured'],
            color=colours, edgecolor='white')
axes[2].set_ylabel('% of Total Fraud')
axes[2].set_title('Fraud Captured per Band', fontweight='bold')
for bar, v in zip(axes[2].patches, band_stats.loc[tiers_present,'pct_fraud_captured']):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{v:.1f}%', ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Score Band Operational Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Worked Example — Scoring a Single Transaction

Demonstrate the additive transparency: every point comes from a specific bin.

In [ ]:
def explain_transaction_score(idx, df_raw, binner, card, y_true=None):
    """
    Print a full scorecard breakdown for a single transaction.
    Shows: feature → raw value → bin → WoE → points.
    """
    row     = df_raw.iloc[[idx]]
    feats   = [f for f in binner.features_ if f in row.columns]
    woe_row = binner.transform(row[feats])
    score   = int(card.predict_score(woe_row)[0])
    prob    = float(card.predict_proba(woe_row)[0])
    tier    = card.score_to_tier(np.array([prob]))[0]

    print('=' * 72)
    print(f'  TRANSACTION SCORECARD BREAKDOWN')
    print('=' * 72)
    if y_true is not None:
        print(f'  Actual label  : {"FRAUD" if y_true.iloc[idx] == 1 else "Legitimate"}')
    print(f'  Total Score   : {score} pts')
    print(f'  P(Fraud)      : {prob:.2%}')
    print(f'  Risk Tier     : {tier}')
    print('-' * 72)
    print(f'  {"Feature":<30} {"Raw Value":<15} {"WoE":>8} {"Points":>8}')
    print(f'  {"":<30} {"":<15} {"(scaled)":>8} {"":>8}')
    print('-' * 72)
    # Offset
    print(f'  {"Base offset (intercept)":<30} {"":>15} {"":>8} {card.offset_:>8}')
    running = card.offset_

    for feat in feats:
        raw_val = float(row[feat].iloc[0])
        woe_val = float(woe_row[feat].iloc[0])
        coef    = card.feature_coefs_.get(feat, 0.0)
        pts     = round(card.B_ * coef * woe_val)
        running += pts
        sign = '+' if pts >= 0 else ''
        print(f'  {feat:<30} {str(round(raw_val, 2)):<15} {woe_val:>8.4f} {sign+str(pts):>8}')

    print('-' * 72)
    print(f'  {"TOTAL SCORE":<30} {"":>15} {"":>8} {running:>8}')
    print('=' * 72)


# Score a high-risk transaction and a low-risk transaction
fraud_idx   = y_test[y_test == 1].index[:1]
legit_idx   = y_test[y_test == 0].index[:1]

print('\n>>> HIGH-RISK TRANSACTION (actual fraud):')
fraud_pos = X_test.index.get_loc(fraud_idx[0])
explain_transaction_score(fraud_pos, X_test, binner, card, y_test)

print('\n>>> LOW-RISK TRANSACTION (actual legitimate):')
legit_pos = X_test.index.get_loc(legit_idx[0])
explain_transaction_score(legit_pos, X_test, binner, card, y_test)

---
## 12. Run Full Training Pipeline & Save Model

In [ ]:
print('Running full train_scorecard() pipeline...')
result = train_scorecard(df, n_bins=8, min_bads=3, lr_C=1.0, min_iv=0.02)

print('\nFinal metrics:')
for k, v in result['metrics'].items():
    print(f'  {k:<28}: {v}')

print('\nSaved to:')
from src.models.scorecard import BINNER_PATH, LR_PATH
print(f'  {BINNER_PATH}')
print(f'  {LR_PATH}')

---
## Summary

| | Scorecard (LR + WoE) |
|---|---|
| **Methodology** | Quantile binning → WoE transformation → LR → PDO scaling |
| **Interpretability** | Fully additive integer points per bin — auditable in a spreadsheet |
| **Primary metric** | AUC-PR (imbalanced data) |
| **Industry metrics** | Gini coefficient, KS statistic (APRA CPG 220 / Basel) |
| **Regulatory alignment** | APRA CPG 220 Tier 2 model — model card in `docs/model_card.md` |
| **Deployment** | `python scripts/train.py --model scorecard` → see FastAPI `/analyse` endpoint |

**Key findings:**
- Weight of Evidence confirmed `high_risk_country`, `velocity_1h`, and `amount_vs_avg_ratio` as the strongest discriminators
- The additive scorecard table enables analyst override and manual audit — critical for AFSL compliance
- Score decile analysis shows fraud is highly concentrated in the top 2 deciles, enabling precise targeting of review workload